# Phase 11 — Hyperparameter Tuning & Temporal Cross-Validation

**Objective:** Tune the strongest models from Phase 10 while preserving the temporal nature of the H&M purchase-prediction problem.

We predict whether a customer will purchase in the **next 30 days**. Therefore, ordinary random K-Fold CV can leak future behavior. This phase uses rolling temporal cutoffs, PR-AUC as the primary metric, and keeps the final Phase 5 test set untouched.

In [ ]:
from pathlib import Path
import json, time, warnings
import numpy as np
import pandas as pd
import joblib

warnings.filterwarnings("ignore")

RANDOM_STATE = 42
HORIZON_DAYS = 30
FAST_MODE = True
MAX_CV_ROWS = 100_000 if FAST_MODE else None

BASE_DIR = Path.cwd()
DATA_DIR = BASE_DIR / "data"
PROCESSED_DIR = DATA_DIR / "processed"
MODELS_DIR = BASE_DIR / "models"
RESULTS_DIR = BASE_DIR / "results"

for p in [PROCESSED_DIR, MODELS_DIR, RESULTS_DIR]:
    p.mkdir(parents=True, exist_ok=True)

print(BASE_DIR)

## 1. Load source data

We return to the transaction-level data because temporal CV must reconstruct features using only information available at each historical cutoff.

Images are not used.

In [ ]:
transactions = pd.read_parquet(PROCESSED_DIR / "transactions_model.parquet")
customers = pd.read_pickle(PROCESSED_DIR / "customers.pkl")
articles = pd.read_pickle(PROCESSED_DIR / "articles.pkl")

transactions["t_dat"] = pd.to_datetime(transactions["t_dat"])

print("Transactions:", transactions.shape)
print("Customers:", customers.shape)
print("Articles:", articles.shape)
print("Dates:", transactions.t_dat.min(), "to", transactions.t_dat.max())

## 2. Temporal feature engineering

For each cutoff:
- features use only `t_dat <= cutoff`
- target uses only `(cutoff, cutoff + 30 days]`

This is the central leakage-prevention rule.

In [ ]:
def build_customer_features(transactions, customers, cutoff_date):
    cutoff_date = pd.Timestamp(cutoff_date)
    hist = transactions[transactions.t_dat <= cutoff_date].copy()
    if hist.empty:
        return pd.DataFrame()

    g = hist.groupby("customer_id")
    f = g.agg(
        total_items=("article_id", "count"),
        purchase_days=("t_dat", "nunique"),
        unique_articles=("article_id", "nunique"),
        total_spend=("price", "sum"),
        avg_price=("price", "mean"),
        median_price=("price", "median"),
        min_price=("price", "min"),
        max_price=("price", "max"),
        channel_1_items=("sales_channel_id", lambda x: (x == 1).sum()),
        channel_2_items=("sales_channel_id", lambda x: (x == 2).sum()),
    ).reset_index()

    f["last_purchase_date"] = g.t_dat.max().values
    f["first_purchase_date"] = g.t_dat.min().values

    f["recency_days"] = (cutoff_date - f.last_purchase_date).dt.days
    f["customer_tenure_days"] = (cutoff_date - f.first_purchase_date).dt.days.clip(lower=1)
    f["purchase_rate"] = f.total_items / f.customer_tenure_days

    for days in [30, 90]:
        recent = hist[hist.t_dat > cutoff_date - pd.Timedelta(days=days)]
        a = recent.groupby("customer_id").agg(
            **{
                f"recent_{days}d_items": ("article_id", "count"),
                f"recent_{days}d_spend": ("price", "sum"),
                f"recent_{days}d_purchase_days": ("t_dat", "nunique"),
                f"recent_{days}d_unique_articles": ("article_id", "nunique"),
            }
        ).reset_index()
        f = f.merge(a, on="customer_id", how="left")

    meta_cols = ["customer_id", "FN", "Active", "club_member_status",
                 "fashion_news_frequency", "age"]
    meta = customers[[c for c in meta_cols if c in customers.columns]]
    f = f.merge(meta, on="customer_id", how="left")

    f["age_missing"] = f["age"].isna().astype(int) if "age" in f else 0
    f["recent_spend_ratio"] = f.recent_30d_spend / f.total_spend.replace(0, np.nan)
    f["recent_items_ratio"] = f.recent_30d_items / f.total_items.replace(0, np.nan)
    f["channel_1_ratio"] = f.channel_1_items / f.total_items.replace(0, np.nan)

    for col in ["total_items", "total_spend", "unique_articles",
                "customer_tenure_days", "recent_30d_items",
                "recent_90d_items", "recent_30d_spend", "recent_90d_spend"]:
        f[f"log1p_{col}"] = np.log1p(f[col].clip(lower=0))

    return f.replace([np.inf, -np.inf], np.nan)


def build_target(transactions, cutoff_date, horizon_days=30):
    cutoff_date = pd.Timestamp(cutoff_date)
    future = transactions[
        (transactions.t_dat > cutoff_date) &
        (transactions.t_dat <= cutoff_date + pd.Timedelta(days=horizon_days))
    ]
    return (future.groupby("customer_id").size().gt(0).astype(int)
            .rename("target").reset_index())

## 3. Create rolling temporal folds

These historical folds are used only for model selection. The Phase 5 validation and test periods remain untouched.

In [ ]:
min_date = transactions.t_dat.min()
max_date = transactions.t_dat.max()

# Historical rolling-origin cutoffs.
cutoffs = [
    min_date + pd.Timedelta(days=365),
    min_date + pd.Timedelta(days=395),
    min_date + pd.Timedelta(days=425),
    min_date + pd.Timedelta(days=455),
]
cutoffs = [c for c in cutoffs
           if c + pd.Timedelta(days=HORIZON_DAYS) < max_date - pd.Timedelta(days=90)]

print("Cutoffs:")
for i, c in enumerate(cutoffs, 1):
    print(i, c.date(), "->", (c + pd.Timedelta(days=HORIZON_DAYS)).date())

In [ ]:
cv_folds = []

for fold_id, cutoff in enumerate(cutoffs, 1):
    X = build_customer_features(transactions, customers, cutoff)
    y = build_target(transactions, cutoff, HORIZON_DAYS)

    d = X.merge(y, on="customer_id", how="left")
    d["target"] = d.target.fillna(0).astype(int)

    if MAX_CV_ROWS and len(d) > MAX_CV_ROWS:
        rng = np.random.RandomState(RANDOM_STATE + fold_id)
        pos, neg = d[d.target == 1], d[d.target == 0]
        npos = min(len(pos), MAX_CV_ROWS // 2)
        nneg = min(len(neg), MAX_CV_ROWS - npos)
        d = pd.concat([
            pos.sample(npos, random_state=rng),
            neg.sample(nneg, random_state=rng)
        ]).sample(frac=1, random_state=rng).reset_index(drop=True)

    cv_folds.append({"fold": fold_id, "cutoff": cutoff, "data": d})
    print(f"Fold {fold_id}: {len(d):,} rows | positive rate={d.target.mean():.4f}")

## 4. Fold-level preprocessing

The preprocessor is fitted separately inside every temporal fold.

- Numerical: median imputation + scaling
- Categorical: constant imputation + one-hot encoding
- Unknown categories: ignored

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

def make_preprocessor(X):
    num = X.select_dtypes(include=[np.number]).columns.tolist()
    cat = X.select_dtypes(include=["object", "category", "bool"]).columns.tolist()

    num_pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="median", add_indicator=True)),
        ("scaler", StandardScaler()),
    ])
    cat_pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="constant", fill_value="Unknown")),
        ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=True)),
    ])

    return ColumnTransformer([
        ("num", num_pipe, num),
        ("cat", cat_pipe, cat),
    ])

## 5. Temporal fold evaluator

In [ ]:
from sklearn.base import clone
from sklearn.metrics import (average_precision_score, roc_auc_score,
                             f1_score, precision_score, recall_score)

def evaluate_fold(model, data, validation_fraction=0.20):
    d = data.copy()
    rng = np.random.RandomState(RANDOM_STATE)

    idx = np.arange(len(d))
    rng.shuffle(idx)
    split = int(len(d) * (1 - validation_fraction))

    tr, va = d.iloc[idx[:split]], d.iloc[idx[split:]]
    Xtr = tr.drop(columns=["customer_id", "target"], errors="ignore")
    Xva = va.drop(columns=["customer_id", "target"], errors="ignore")
    ytr, yva = tr.target, va.target

    prep = make_preprocessor(Xtr)
    Xtr_t = prep.fit_transform(Xtr)
    Xva_t = prep.transform(Xva)

    m = clone(model)
    start = time.perf_counter()
    m.fit(Xtr_t, ytr)
    fit_time = time.perf_counter() - start

    p = m.predict_proba(Xva_t)[:, 1]
    pred = (p >= 0.5).astype(int)

    return {
        "pr_auc": average_precision_score(yva, p),
        "roc_auc": roc_auc_score(yva, p),
        "f1": f1_score(yva, pred, zero_division=0),
        "precision": precision_score(yva, pred, zero_division=0),
        "recall": recall_score(yva, pred, zero_division=0),
        "fit_time_sec": fit_time,
    }

## 6. Candidate models

We tune the main nonlinear models from Phase 10:
- Random Forest
- Extra Trees
- XGBoost, when available

In [ ]:
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier

try:
    from xgboost import XGBClassifier
    XGB_AVAILABLE = True
except Exception:
    XGB_AVAILABLE = False

print("XGBoost available:", XGB_AVAILABLE)

## 7. Hyperparameter spaces

The search focuses on parameters controlling:
- model complexity
- regularization
- sampling
- learning rate
- number of trees

In [ ]:
rf_space = [
    {"n_estimators": n, "max_depth": d, "min_samples_leaf": l, "max_features": mf}
    for n in [100, 200, 300]
    for d in [8, 12, 16, None]
    for l in [5, 20, 50]
    for mf in ["sqrt", 0.5]
]

et_space = [
    {"n_estimators": n, "max_depth": d, "min_samples_leaf": l, "max_features": mf}
    for n in [100, 200, 300]
    for d in [8, 12, 16, None]
    for l in [5, 20, 50]
    for mf in ["sqrt", 0.5]
]

xgb_space = [
    {"n_estimators": n, "max_depth": d, "learning_rate": lr,
     "min_child_weight": mcw, "subsample": ss,
     "colsample_bytree": cs, "reg_lambda": reg}
    for n in [150, 300, 500]
    for d in [3, 5, 7]
    for lr in [0.03, 0.05, 0.10]
    for mcw in [1, 10, 20]
    for ss in [0.7, 0.9]
    for cs in [0.7, 0.9]
    for reg in [1.0, 10.0]
]

rng = np.random.RandomState(RANDOM_STATE)

def sample_configs(space, n):
    ids = rng.choice(len(space), min(n, len(space)), replace=False)
    return [space[i] for i in ids]

rf_configs = sample_configs(rf_space, 6 if FAST_MODE else 15)
et_configs = sample_configs(et_space, 6 if FAST_MODE else 15)
xgb_configs = sample_configs(xgb_space, 6 if FAST_MODE else 15)

print(len(rf_configs), len(et_configs), len(xgb_configs))

## 8. Model factory

In [ ]:
def make_model(name, p):
    if name == "Random Forest":
        return RandomForestClassifier(
            **p, class_weight="balanced_subsample",
            n_jobs=-1, random_state=RANDOM_STATE
        )
    if name == "Extra Trees":
        return ExtraTreesClassifier(
            **p, class_weight="balanced",
            n_jobs=-1, random_state=RANDOM_STATE
        )
    if name == "XGBoost":
        return XGBClassifier(
            **p, tree_method="hist", eval_metric="logloss",
            n_jobs=-1, random_state=RANDOM_STATE
        )
    raise ValueError(name)

## 9. Run temporal hyperparameter search

**Primary metric:** mean PR-AUC across temporal folds.

The final Phase 5 validation and test sets are not used.

In [ ]:
def temporal_search(name, configs):
    rows = []

    for cid, params in enumerate(configs, 1):
        print(f"{name}: configuration {cid}/{len(configs)}")

        for fold in cv_folds:
            start = time.perf_counter()
            metrics = evaluate_fold(make_model(name, params), fold["data"])
            total = time.perf_counter() - start

            rows.append({
                "model": name,
                "config_id": cid,
                "fold": fold["fold"],
                "cutoff": fold["cutoff"],
                "params": json.dumps(params),
                "total_time_sec": total,
                **metrics
            })

        temp = pd.DataFrame(rows)
        last = temp[temp.config_id == cid]
        print("  mean PR-AUC:", round(last.pr_auc.mean(), 5))

    return pd.DataFrame(rows)

rf_results = temporal_search("Random Forest", rf_configs)
et_results = temporal_search("Extra Trees", et_configs)

if XGB_AVAILABLE:
    xgb_results = temporal_search("XGBoost", xgb_configs)
else:
    xgb_results = pd.DataFrame()

search_results = pd.concat([rf_results, et_results, xgb_results], ignore_index=True)
search_results.to_csv(RESULTS_DIR / "phase11_temporal_cv_results.csv", index=False)

print("Search complete:", search_results.shape)

## 10. Aggregate configurations

In [ ]:
summary = (
    search_results.groupby(["model", "config_id", "params"], as_index=False)
    .agg(
        mean_pr_auc=("pr_auc", "mean"),
        std_pr_auc=("pr_auc", "std"),
        mean_roc_auc=("roc_auc", "mean"),
        mean_f1=("f1", "mean"),
        mean_precision=("precision", "mean"),
        mean_recall=("recall", "mean"),
        mean_fit_time_sec=("fit_time_sec", "mean"),
    )
    .sort_values(["mean_pr_auc", "std_pr_auc"], ascending=[False, True])
)

display(summary.head(15))
summary.to_csv(RESULTS_DIR / "phase11_hyperparameter_search.csv", index=False)

## 11. Select the best configuration

Selection is based on:
1. highest mean temporal PR-AUC
2. lower temporal variation as a tie-breaker

The test set is still untouched.

In [ ]:
best = summary.iloc[0]
best_model_name = best.model
best_params = json.loads(best.params)

print("BEST MODEL:", best_model_name)
print("MEAN PR-AUC:", round(best.mean_pr_auc, 5))
print("STD PR-AUC:", round(best.std_pr_auc, 5))
print(json.dumps(best_params, indent=2))

## 12. Inspect fold-by-fold stability

In [ ]:
best_folds = search_results[
    (search_results.model == best_model_name) &
    (search_results.config_id == best.config_id)
].sort_values("fold")

display(best_folds[[
    "fold", "cutoff", "pr_auc", "roc_auc",
    "f1", "precision", "recall"
]])

## 13. Compare tuned models against their searched baseline configurations

In [ ]:
baseline = (
    search_results.groupby(["model", "fold"], as_index=False)
    .first()
    .groupby("model")
    .agg(
        baseline_mean_pr_auc=("pr_auc", "mean"),
        baseline_mean_roc_auc=("roc_auc", "mean")
    )
    .reset_index()
)

tuned = (
    summary.groupby("model", as_index=False)
    .first()
    [["model", "mean_pr_auc", "mean_roc_auc"]]
)

comparison = tuned.merge(baseline, on="model", how="left")
comparison["pr_auc_change"] = (
    comparison.mean_pr_auc - comparison.baseline_mean_pr_auc
)
comparison["roc_auc_change"] = (
    comparison.mean_roc_auc - comparison.baseline_mean_roc_auc
)

display(comparison)
comparison.to_csv(RESULTS_DIR / "phase11_tuned_vs_baseline.csv", index=False)

## 14. Train the selected tuned model on the full Phase 5 training set

This prepares the frozen model for Phase 12.

The Phase 5 validation and test sets are **not evaluated here**. They remain holdouts for final evaluation.

In [ ]:
train_df = pd.read_parquet(PROCESSED_DIR / "train_phase5.parquet")
y_train = pd.read_parquet(PROCESSED_DIR / "y_train_phase5.parquet").squeeze()

X_train = train_df.drop(columns=["customer_id"], errors="ignore")

final_preprocessor = make_preprocessor(X_train)

start = time.perf_counter()
X_train_t = final_preprocessor.fit_transform(X_train)

final_model = make_model(best_model_name, best_params)
final_model.fit(X_train_t, y_train)
training_time = time.perf_counter() - start

print("Selected:", best_model_name)
print("Training time:", round(training_time, 2), "sec")
print("Transformed shape:", X_train_t.shape)

## 15. Save artifacts

In [ ]:
model_file = MODELS_DIR / (
    best_model_name.lower().replace(" ", "_") +
    "_phase11_tuned.joblib"
)

joblib.dump(final_preprocessor, MODELS_DIR / "preprocessor_phase11_tuned.joblib")
joblib.dump(final_model, model_file)

config = {
    "phase": 11,
    "model": best_model_name,
    "best_params": best_params,
    "mean_temporal_pr_auc": float(best.mean_pr_auc),
    "std_temporal_pr_auc": float(best.std_pr_auc),
    "mean_temporal_roc_auc": float(best.mean_roc_auc),
    "selection_metric": "mean_temporal_PR_AUC",
    "horizon_days": HORIZON_DAYS,
    "phase5_validation_used_for_tuning": False,
    "phase5_test_used_for_tuning": False,
}

with open(RESULTS_DIR / "phase11_best_params.json", "w") as f:
    json.dump(config, f, indent=2)

print("Saved:", model_file)

## 16. Leakage audit

| Check | Expected |
|---|---|
| Features use only pre-cutoff transactions | ✅ |
| Target uses only next 30 days | ✅ |
| Preprocessor fitted inside each fold | ✅ |
| Phase 5 validation used for tuning | ❌ |
| Phase 5 test used for tuning | ❌ |
| Final tuned model trained on Phase 5 train | ✅ |

### Placement-ready statement

> **Performed time-aware hyperparameter optimization using rolling-origin temporal cross-validation, selecting the model based on mean PR-AUC and temporal stability while preventing future-information leakage.**

## Next: Phase 12

**Final Evaluation, Threshold Optimization, Error Analysis & Explainability**

We will finally use the untouched validation/test sets to:
- choose the operating threshold on validation,
- report final test metrics,
- compare against the baseline,
- analyze false positives/false negatives,
- inspect feature importance,
- quantify improvement,
- and freeze the final project model.